In [17]:
import requests
import pandas as pd
import kagglehub
from recognition_evaluators import evaluate_tm_df
from util import dictcount

In [18]:
BASE_URL = "https://deepcode.ci.nsu.ru"
api_key = "sk-d03991b7df2a4bd0b306fe9dbb2d6389"
headers = {"Authorization": f"Bearer {api_key}"}

MODEL_ID = "Qwen3.8-27B"
print("Model:", MODEL_ID)


Model: Qwen3.8-27B


In [19]:
multiclass_dataset_path = kagglehub.dataset_download("sagarikashreevastava/cognitive-distortion-detetction-dataset")
multiclass_dataset_file_path = multiclass_dataset_path + "/Annotated_data.csv"
df = pd.read_csv(multiclass_dataset_file_path)
df = df.drop("Id_Number", axis=1)
df

,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN
3,My parents got divorced in 2004. My mother has...,NaN,No Distortion,NaN
4,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning
...,...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,NaN,No Distortion,NaN
2526,I am 21 female and have not had any friends fo...,Now I am at university my peers around me all ...,Overgeneralization,NaN
2527,From the U.S.: My brother is 19 years old and ...,He claims he’s severely depressed and has outb...,Mental filter,Mind Reading
2528,From the U.S.: I am a 21 year old woman who ha...,NaN,No Distortion,NaN


In [20]:
all_dists = {}
for _, row in df.iterrows():
    primary_distortion = row.iloc[2]
    secondary_distortion = row.iloc[3] if pd.notna(row.iloc[3]) else None
    dictcount(all_dists, primary_distortion)
    if secondary_distortion is not None:
        dictcount(all_dists, secondary_distortion)
del all_dists["No Distortion"]

labels = sorted(all_dists)
all_dists

{'Personalization': 202,
 'Labeling': 203,
 'Emotional Reasoning': 169,
 'Fortune-telling': 210,
 'Magnification': 245,
 'Mind Reading': 295,
 'All-or-nothing thinking': 126,
 'Overgeneralization': 277,
 'Mental filter': 151,
 'Should statements': 135}

In [21]:
def evaluate_specific_model(model, text, dists, debug=False):
    global last_response
    query = (
        f'You are professional psycho-therapist experienced in cognitive-behavioral therapy. '
        f'You can label texts witn none or some of cognitive distortoions, represented by the following labels: {str(dists)}. '
        f'When labeling, return only JSON array in square brackets of strings in double quotes representing the labels. '
        f'Label this text for presense or absence of any cognitive distortions fom given list: "{text}"'
    )
    if debug:
        print(query)
    response = requests.post(
        BASE_URL + "/api/chat/completions",
        headers=headers,
        json={
            "model": model,
            "messages": [{"role": "user", "content": query}],
            "temperature": 0.0,
            "chat_template_kwargs": {"enable_thinking": False},
            "max_tokens": 4096,
            "stream": False
        },
        timeout=300
    )
    response.raise_for_status()
    last_response = response.json()
    return last_response["choices"][0]["message"]["content"]


def llm_evaluator_top1(all_metrics, model, text, threshold, top=1):
    rep = evaluate_specific_model(model, text, dists=all_metrics, debug=False)
    rep_list = []
    for metric in all_metrics:
        if metric in rep:
            rep_list.append(metric)
    return {value: 1.0 for index, value in enumerate(rep_list[:top])}

In [22]:
predictions = []


In [24]:
for index, row in df.iloc[len(predictions):].iterrows():
    text = row.iloc[1] if pd.notna(row.iloc[1]) else row.iloc[0]
    predictions.append(llm_evaluator_top1(all_dists, MODEL_ID, text, None))


def stored_evaluator(all_metrics, predictions_iter, text, threshold, min_count):
    return next(predictions_iter)

precision, recall, f1, accuracy = evaluate_tm_df(
    df, iter(predictions), stored_evaluator, None,
    all_metrics=all_dists, encode_spaces=False, debug=False
)
print("F1:", f1)


F1: {'Personalization': 0.5989847715736041, 'Labeling': 0.5851063829787234, 'Emotional Reasoning': 0.2222222222222222, 'Fortune-telling': 0.5344827586206896, 'Magnification': 0.7272727272727273, 'Mind Reading': 0.7521367521367521, 'All-or-nothing thinking': 0.32116788321167883, 'Overgeneralization': 0.5891472868217055, 'Mental filter': 0.25, 'Should statements': 0.7708333333333333}


In [25]:
macro_f1 = sum(f1.values()) / len(f1)
print("Rows:", len(df))
print("Macro F1:", macro_f1)

true_labels = []
for _, row in df.iterrows():
    labels_for_row = []
    if row.iloc[2] != "No Distortion":
        labels_for_row.append(row.iloc[2])
    if pd.notna(row.iloc[3]) and row.iloc[3] != "No Distortion":
        labels_for_row.append(row.iloc[3])
    true_labels.append(labels_for_row)

results_df = pd.DataFrame({
    "text": [row.iloc[1] if pd.notna(row.iloc[1]) else row.iloc[0] for _, row in df.iterrows()],
    "true_labels": true_labels,
    "predicted_labels": [list(prediction) for prediction in predictions]
})

metrics_df = pd.DataFrame({
    "label": labels,
    "precision": [precision[label] for label in labels],
    "recall": [recall[label] for label in labels],
    "f1": [f1[label] for label in labels],
    "accuracy": [accuracy[label] for label in labels]
})
display(results_df)
display(metrics_df)


Rows: 2530
Macro F1: 0.5351354118171436


,text,true_labels,predicted_labels
0,The voice are always fimilar (someone she know...,[Personalization],[]
1,I feel trapped inside my disgusting self and l...,"[Labeling, Emotional Reasoning]",[Labeling]
2,So I’ve been dating on and off this guy for a...,[],[Emotional Reasoning]
3,My parents got divorced in 2004. My mother has...,[],[]
4,I refused to go because I didn’t know if it wa...,"[Fortune-telling, Emotional Reasoning]",[Fortune-telling]
...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,[],[All-or-nothing thinking]
2526,Now I am at university my peers around me all ...,[Overgeneralization],[Overgeneralization]
2527,He claims he’s severely depressed and has outb...,"[Mental filter, Mind Reading]",[]
2528,From the U.S.: I am a 21 year old woman who ha...,[],[]


,label,precision,recall,f1,accuracy
0,All-or-nothing thinking,0.191304,1.0,0.321168,0.008696
1,Emotional Reasoning,0.125000,1.0,0.222222,0.001581
2,Fortune-telling,0.364706,1.0,0.534483,0.049012
3,Labeling,0.413534,1.0,0.585106,0.043478
4,Magnification,0.571429,1.0,0.727273,0.009486
5,Mental filter,0.142857,1.0,0.250000,0.001581
6,Mind Reading,0.602740,1.0,0.752137,0.052174
7,Overgeneralization,0.417582,1.0,0.589147,0.030040
8,Personalization,0.427536,1.0,0.598985,0.023320
9,Should statements,0.627119,1.0,0.770833,0.014625
